导入题目数据，合并

In [1]:

import pandas as pd

train = pd.read_csv("data/train.csv")
test  = pd.read_csv("data/test.csv")

train['train_test'] = 1  # 添加标识列
test['train_test'] = 0
all_data=pd.concat([train, test])  # 合并数据集



In [2]:
print(all_data.info())  #查看数据集的基本信息
print(all_data.head())  #查看数据集的前几行
print(all_data.columns)

<class 'pandas.core.frame.DataFrame'>
Index: 57480 entries, 0 to 2
Data columns (total 10 columns):
 #   Column          Non-Null Count  Dtype  
---  ------          --------------  -----  
 0   id              57480 non-null  int64  
 1   model_a         57477 non-null  object 
 2   model_b         57477 non-null  object 
 3   prompt          57480 non-null  object 
 4   response_a      57480 non-null  object 
 5   response_b      57480 non-null  object 
 6   winner_model_a  57477 non-null  float64
 7   winner_model_b  57477 non-null  float64
 8   winner_tie      57477 non-null  float64
 9   train_test      57480 non-null  int64  
dtypes: float64(3), int64(2), object(5)
memory usage: 4.8+ MB
None
       id             model_a              model_b  \
0   30192  gpt-4-1106-preview           gpt-4-0613   
1   53567           koala-13b           gpt-4-0613   
2   65089  gpt-3.5-turbo-0613       mistral-medium   
3   96401    llama-2-13b-chat  mistral-7b-instruct   
4  198779           koa

In [3]:
from sklearn.feature_extraction.text import TfidfVectorizer
import numpy as np

# 初始化TF-IDF向量化器，限制特征数量为500
vectorizer_prompt=TfidfVectorizer(max_features=500)  # 限制特征数量
vectorizer_response_a=TfidfVectorizer(max_features=500)
vectorizer_response_b=TfidfVectorizer(max_features=500)

# 对每个文本列进行TF-IDF向量化
#.fit_transform方法会返回一个稀疏矩阵，使用toarray()将其转换为密集矩阵
temp_prompt=vectorizer_prompt.fit_transform(train['prompt'])
temp_response_a=vectorizer_response_a.fit_transform(train['response_a'])
temp_response_b=vectorizer_response_b.fit_transform(train['response_b'])



In [4]:
# 提取需要的特征列并转换为NumPy数组
def get_winner(row):
    if row['winner_model_a'] == 1:
        return 0  # 模型A获胜
    elif row['winner_model_b'] == 1:
        return 1  # 模型B获胜
    elif row['winner_tie'] == 1:
        return 2  # 平局
    else:
        return -1 # 无效数据

train['winner']=train.apply(get_winner,axis=1) # 将结果存储在新的列中
train_y=train['winner'].values  # 提取标签列作为训练目标

In [5]:
# 选择特征
# 将三个文本列的TF-IDF特征合并为一个特征矩阵
train_X=np.concatenate((temp_prompt.toarray(),temp_response_a.toarray()
                        ,temp_response_b.toarray()),axis=1)  

In [ ]:
# 逻辑回归
from sklearn.linear_model import LogisticRegression
from datetime import datetime

start_time=datetime.now()  # 记录开始时间
# 初始化逻辑回归模型，设置最大迭代次数为1000，使用多项式损失函数和saga优化器
model=LogisticRegression(max_iter=1000,multi_class='multinomial',solver='saga')  
model.fit(train_X,train_y)  # 训练模型
end_time=datetime.now()  # 记录结束时间
print(f"训练时间: {end_time - start_time}") 

训练时间: 0:00:00


In [7]:
from sklearn.model_selection import train_test_split # 导入数据集划分函数
from sklearn.metrics import confusion_matrix, precision_score, recall_score # 导入评估指标函数

# 划分训练集和验证集，测试集占20%
train_X_train,train_X_val,train_y_train,train_y_val=train_test_split(train_X,train_y,test_size=0.2,random_state=42)
start_time=datetime.now()  # 记录开始时间

value_y_predict=model.predict(train_X_val)  # 在验证集上进行预测
print('预测',value_y_predict)
print('真实',train_y_val)

value_y_probabilities=model.predict_proba(train_X_val)  # 获取预测概率
print('预测概率',value_y_probabilities)

# 计算混淆矩阵、精确率和召回率
cm=confusion_matrix(train_y_val,value_y_predict)
print('混淆矩阵:\n',cm)

# 模型精确率
score=model.score(train_X_val,train_y_val)
print('模型精确率:',score)

# 宏观和微观平均精度和召回率
macro_precision=precision_score(train_y_val,value_y_predict,average='macro')
macro_recall=recall_score(train_y_val,value_y_predict,average='macro')
micro_precision=precision_score(train_y_val,value_y_predict,average='micro')
micro_recall=recall_score(train_y_val,value_y_predict,average='micro')
print('宏观平均精确率:',macro_precision)
print('宏观平均召回率:',macro_recall)
print('微观平均精确率:',micro_precision)
print('微观平均召回率:',micro_recall)

end_time=datetime.now()  # 记录结束时间
print(f"评估时间: {end_time - start_time}")

NotFittedError: This LogisticRegression instance is not fitted yet. Call 'fit' with appropriate arguments before using this estimator.

In [ ]:
from sklearn.metrics import log_loss # 导入对数损失函数
model_log_loss=log_loss(train_y_val,value_y_probabilities)  # 计算对数损失
print('模型对数损失:',model_log_loss)  



模型对数损失: 1.017427583159


In [ ]:
# 对测试集进行相同的特征处理
temp_test_prompt=vectorizer_prompt.transform(test['prompt'])
temp_response_a=vectorizer_response_a.transform(test['response_a'])
temp_response_b=vectorizer_response_b.transform(test['response_b'])

# 将测试集的TF-IDF特征合并为一个特征矩阵
test_X=np.concatenate((temp_test_prompt.toarray(),temp_response_a.toarray()
                        ,temp_response_b.toarray()),axis=1)
value_test_y_probabilities=model.predict_proba(test_X)  # 获取测试集的预测概率
print('测试集预测概率',value_test_y_probabilities)


测试集预测概率 [[0.13176426 0.2584409  0.60979484]
 [0.30718164 0.36532827 0.32749009]
 [0.29710546 0.35404737 0.34884717]]


In [ ]:
output=pd.DataFrame({'id':test.id,
                     'winner_model_a':value_test_y_probabilities[:,0],
                     'winner_model_b':value_test_y_probabilities[:,1],
                     'winner_tie':value_test_y_probabilities[:,2]})
output.to_csv('submission.csv',index=False)  # 将结果保存为CSV文件